# Fine-tune Ettin on CUAD (Kaggle, T4 x2)

Trains an Ettin encoder for extractive QA on CUAD, chooses the epoch by validation AUPR and
scores the official test split once per seed.

Before running: add this repository as a Kaggle dataset (`make kaggle-zip` builds the
archive), set the accelerator to *GPU T4 x2* and turn on Internet access.

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

WORK = Path("/kaggle/working/cuad-encoder")
roots = [p.parent.parent for p in Path("/kaggle/input").rglob("cuad_encoder/__init__.py")]
if roots:
    shutil.copytree(roots[0], WORK, dirs_exist_ok=True)
else:
    archive = next(Path("/kaggle/input").rglob("cuad-encoder*.zip"))
    zipfile.ZipFile(archive).extractall(WORK)
os.chdir(WORK)
print(sorted(p.name for p in WORK.iterdir()))

In [ ]:
!pip -q install "transformers>=5.13,<5.16" "tokenizers<0.23" "datasets>=2.19" "accelerate>=0.30"
!nvidia-smi --query-gpu=index,name,memory.total --format=csv,noheader

## Data and features

The 400M model needs gradient checkpointing to fit a T4.

In [ ]:
MODEL = "jhu-clsp/ettin-encoder-150m"
GRAD_CKPT = MODEL.endswith("400m")
SEEDS = [3407, 42, 1234]
FEATURES = f"data/features/{MODEL.split('/')[-1]}_w1024_s256_multi_neg1"

!python -m cuad_encoder.data
!python -m cuad_encoder.features --tokenizer {MODEL}

## Train, choose the epoch, score the test split

In [ ]:
import torch

from cuad_encoder.pipeline import Config, run_seed, summarize

cfg = Config(features=FEATURES, model=MODEL, grad_ckpt=GRAD_CKPT, n_gpus=torch.cuda.device_count())
rows = [run_seed(cfg, seed) for seed in SEEDS]
summarize(rows, Path(cfg.results) / f"{cfg.feature_tag}_e{cfg.max_epochs}v_seeds_summary.csv")

## Thresholds and figures for the first seed

In [ ]:
import json

from IPython.display import Image, display

run = cfg.run_name(SEEDS[0])
epoch = json.loads(Path(f"runs/{run}/selection.json").read_text())["selected"]["epoch"]

!python -m cuad_encoder.thresholds --val-preds results/{run}_VAL_ep{epoch:g}_preds.jsonl \
    --test-preds results/{run}_preds.jsonl --out results/{run}_operating_point.json
!python -m cuad_encoder.figures draw --out results/figures/{run} \
    --system "{run}=results/{run}_by_type.csv" --preds "{run}=results/{run}_preds.jsonl" \
    --log-history runs/{run}/log_history.json

for png in sorted(Path(f"results/figures/{run}").glob("*.png")):
    display(Image(str(png)))

The reports, predictions and figures are in `results/`, the selected model of each seed in
`runs/<run>/final/`; both can be downloaded from the Output tab.